In [1]:
import numpy as np
import pyvista as pv
from scipy.spatial import KDTree
import sys
sys.path.append("..")
from error_func import error

case_name = "case2"
mesh_name = "coil_box"
solver1 = "ngsolve"
solver2 = "comsol"

sol_ngsolve = np.load(f"../../output/{case_name}/gauss/{case_name}_{solver1}.npy")
sol_comsol = np.genfromtxt(f"../../output/{case_name}/gauss/{case_name}_comsol.txt", delimiter = " ")

print(sol_ngsolve.shape)
print(sol_comsol.shape)

print(sol_ngsolve.shape)
print(sol_comsol.shape)


(69629, 10)
(69629, 11)
(69629, 10)
(69629, 11)


In [2]:
print("Min and max X coord:")
print(np.min(sol_ngsolve[:, 0:3]))
print(np.max(sol_ngsolve[:, 0:3]))

print("Min and max Y coord:")
print(np.min(sol_ngsolve[:, 0:3]))
print(np.max(sol_ngsolve[:, 0:3]))

print("Min and max Z coord:")
print(np.min(sol_ngsolve[:, 0:3]))
print(np.max(sol_ngsolve[:, 0:3]))

print("")

print("Min and max X coord:")
print(np.min(sol_comsol[:, 0:3]))
print(np.max(sol_comsol[:, 0:3]))

print("Min and max Y coord:")
print(np.min(sol_comsol[:, 0:3]))
print(np.max(sol_comsol[:, 0:3]))

print("Min and max Z coord:")
print(np.min(sol_comsol[:, 0:3]))
print(np.max(sol_comsol[:, 0:3]))

Min and max X coord:
-0.099551605255125
0.09795298646876001
Min and max Y coord:
-0.099551605255125
0.09795298646876001
Min and max Z coord:
-0.099551605255125
0.09795298646876001

Min and max X coord:
-0.09955160525513063
0.0979529864687601
Min and max Y coord:
-0.09955160525513063
0.0979529864687601
Min and max Z coord:
-0.09955160525513063
0.0979529864687601


In [3]:
tree = KDTree(sol_ngsolve[:, 0:3])
distances, indices = tree.query(sol_comsol[:, 0:3])

max_dist = np.max(distances)
print(f"Maximum alignment error (distance): {max_dist:.6e}")
if max_dist > 1e-4:
    print("Warning: Large distance detected. Are the geometries identical?")


sol_ngsolve_reordered = sol_ngsolve[indices, :]

np.save(f"../../output/{case_name}/gauss/{case_name}_{solver1}_reordered_to_{solver2}.npy", sol_ngsolve_reordered)

Maximum alignment error (distance): 1.344667e-13


In [4]:
sol_ngsolve = np.load(f"../../output/{case_name}/gauss/{case_name}_{solver1}_reordered_to_{solver2}.npy")
elec_pot_ngsolve = sol_ngsolve[:, 9]
mag_flux_ngsolve = sol_ngsolve[:, 6:9]
mag_vec_ngsolve = sol_ngsolve[:, 3:6]

sol_comsol = np.genfromtxt(f"../../output/{case_name}/gauss/{case_name}_comsol.txt", delimiter = " ")
elec_pot_comsol = sol_comsol[:, 9]
mag_flux_comsol = sol_comsol[:, 6:9]
mag_vec_comsol = sol_comsol[:, 3:6]

# print(elec_pot_ngsolve.shape)
# print(elec_pot_comsol.shape)

print(mag_flux_ngsolve.shape)
print(mag_flux_comsol.shape)

print(mag_vec_ngsolve.shape)
print(mag_vec_comsol.shape)

(69629, 3)
(69629, 3)
(69629, 3)
(69629, 3)


In [5]:
mesh = sol_comsol.copy()

In [6]:
print(f"Coordinate errors between {solver1} and {solver2}:")

mesh = error(sol=sol_ngsolve[:, 0:3], sol_ref=sol_comsol[:, 0:3], 
             eps = 1e-12, mesh=mesh, tag="vector", save_tag=None)

Coordinate errors between ngsolve and comsol:

  * Max. absolute error in x direction  : 9.454e-14.
  * Avg. absolute error in x direction  : 2.606e-14.

  * Max. relative error in x direction : 3.968e-06 %.
  * Avg. relative error in x direction : 3.634e-10 %.

  * Max. absolute error in y direction  : 9.714e-14.
  * Avg. absolute error in y direction  : 2.783e-14.

  * Max. relative error in y direction : 1.209e-06 %.
  * Avg. relative error in y direction : 2.947e-10 %.

  * Max. absolute error in z direction  : 9.581e-14.
  * Avg. absolute error in z direction  : 2.571e-14.

  * Max. relative error in z direction : 4.411e-06 %.
  * Avg. relative error in z direction : 8.938e-10 %.


In [7]:
print(f"Magnetic flux density errors between {solver1} and {solver2}:")

mesh = error(sol=mag_flux_ngsolve, sol_ref=mag_flux_comsol, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag=None)

Magnetic flux density errors between ngsolve and comsol:

  * Max. absolute error in x direction  : 4.188e+04.
  * Avg. absolute error in x direction  : 9.011e+02.

  * Max. relative error in x direction : 2.325e+05 %.
  * Avg. relative error in x direction : 4.392e+01 %.

  * Max. absolute error in y direction  : 7.062e+03.
  * Avg. absolute error in y direction  : 3.522e+02.

  * Max. relative error in y direction : 8.055e+05 %.
  * Avg. relative error in y direction : 8.008e+01 %.

  * Max. absolute error in z direction  : 4.702e+04.
  * Avg. absolute error in z direction  : 5.068e+03.

  * Max. relative error in z direction : 1.770e+06 %.
  * Avg. relative error in z direction : 2.326e+02 %.


/home/wiera/Documents/EM_simulation/scripts/case2/../error_func.py:62: RuntimeWarning: divide by zero encountered in divide
  rel_error_dir = np.where(denom > eps, num * 100 / denom, np.nan)
/home/wiera/Documents/EM_simulation/scripts/case2/../error_func.py:62: RuntimeWarning: invalid value encountered in divide
  rel_error_dir = np.where(denom > eps, num * 100 / denom, np.nan)


In [8]:
print(f"Magnetic vector potential errors between {solver1} and {solver2}:")

mesh = error(sol=mag_vec_ngsolve, sol_ref=mag_vec_comsol, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag=None)

Magnetic vector potential errors between ngsolve and comsol:

  * Max. absolute error in x direction  : 1.802e+14.
  * Avg. absolute error in x direction  : 1.188e+12.

  * Max. relative error in x direction : 2.978e+16 %.
  * Avg. relative error in x direction : 9.743e+12 %.

  * Max. absolute error in y direction  : 1.857e+14.
  * Avg. absolute error in y direction  : 1.781e+12.

  * Max. relative error in y direction : 1.207e+15 %.
  * Avg. relative error in y direction : 5.540e+11 %.

  * Max. absolute error in z direction  : 1.756e+14.
  * Avg. absolute error in z direction  : 9.658e+11.

  * Max. relative error in z direction : 3.884e+16 %.
  * Avg. relative error in z direction : 9.427e+12 %.
